In [1]:
import pandas as pd
import numpy as np
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

In [2]:
ke_df=pd.read_excel("Buffalo_NY_OB_KINGElvis.xlsx",sheet_name='Elvis_Review')
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['HAVE_5_MIN_FOR_SURVECode']==1]

df=pd.read_csv('elvisbuffalony2024obweekday_export_odbc.csv')

In [3]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [5]:
origin_destin_transport_column_checks=['origintransport','destintransport']
origin_destin_transport_column=check_all_characters_present(df,origin_destin_transport_column_checks)
origin_destin_transport_column

['ORIGIN_TRANSPORT', 'DESTIN_TRANSPORT']

In [7]:
ke_df=pd.merge(ke_df,df[['id',*origin_destin_transport_column]],on='id',how='left',indicator=True)
ke_df['Matched'] = (ke_df['_merge'] == 'both').astype(int)
ke_df.drop(columns=['_merge'],inplace=True)

In [8]:
ke_df.head()

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,RECORD_INFO,ORIGIN_TRANSPORT_y,DESTIN_TRANSPORT_y,Matched
0,2024-09-25,5907,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Walk,Walk,1
1,2024-09-25,5914,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Walk,Walk,1
2,2024-09-25,5917,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Walk,Walk,1
3,2024-09-25,5920,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Walk,Walk,1
4,2024-09-25,5922,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Walk,Walk,1


In [15]:
filtered_df = ke_df.query(
    "ORIGIN_TRANSPORT_y.str.contains(r'\\bOther\\b', case=False) or DESTIN_TRANSPORT_y.str.contains(r'\\bOther\\b', case=False)", 
    engine='python'
)

In [16]:
filtered_df.head()

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,RECORD_INFO,ORIGIN_TRANSPORT_y,DESTIN_TRANSPORT_y,Matched
210,2024-09-25,6514,HereAPI,Sneh,Remove,NaN,O is above 50 miles,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Walk,Other,1
241,2024-09-25,6573,HereAPI,Sneh,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other,Walk,1
309,2024-09-25,6735,HereAPI,Kesar,Remove,NaN,use of transit not necessary,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other,Walk,1
656,2024-09-25,7542,HereAPI,muskan,Use,NaN,NaN,Elvis_Review,Elvis_Review,O= Lovejoy and Bailey,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Walk,Other,1
710,2024-09-25,7670,HereAPI,muskan,Use,NaN,NaN,Elvis_Review,Elvis_Review,no streets/address,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other,Walk,1


In [17]:
filtered_df.shape

(49, 47)